# Q3 — ESG Message Triage using LLM
## Assessment 3 — Cloud Platforms

**Instructions:**
1. Run each cell in order (Shift + Enter)
2. Add your OpenAI API key in Cell 2
3. Take screenshots of each output
4. Upload this notebook to GitHub

## STEP 1 — Install Libraries

In [ ]:
# Install required libraries
!pip install openai -q
!pip install transformers -q
!pip install torch -q
print('✅ All libraries installed!')

## STEP 2 — Add Your OpenAI API Key

In [ ]:
# ============================================================
# ADD YOUR API KEY HERE
# ============================================================
OPENAI_API_KEY = "YOUR_API_KEY_HERE"   # <-- yahan apni key likho

print('✅ API Key set!')

## STEP 3 — Q3(a): Revised Prompt + GPT-4o Triage

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

# ============================================================
# REVISED SYSTEM PROMPT (improved from original)
# ============================================================
SYSTEM_PROMPT = """
You are an expert ESG (Environmental, Social, and Governance) triage assistant
working for a large organisation. Your role is to analyse employee-submitted
operational messages and classify them with precision.

For every message, return a JSON object ONLY.
No explanation, no markdown fences, no extra text.

Classification rules:
- issue_category: ONE of [Energy, Water, Waste, Air Quality,
  Sustainability Policy, Procurement, Accessibility, Wellbeing, Governance, Other]

- urgency:
    CRITICAL = immediate risk to health, safety or major legal/compliance breach
    HIGH     = action required within 24 hours
    MEDIUM   = action required within 1 week
    LOW      = informational or minor concern

- sentiment: POSITIVE | NEUTRAL | NEGATIVE
- followup_required: Y or N
- recommended_team: specific team name
- escalation_reason: clear reason, or 'None' if LOW
- data_sensitivity_risk: LOW | MEDIUM | HIGH
- brief_summary: ONE sentence max
- suggested_action: ONE concrete next step

Return ONLY valid JSON. No preamble. No explanation.
"""

# ============================================================
# 5 ESG TEST MESSAGES
# ============================================================
messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "The air conditioning is running overnight in an empty office.",
    "I want to report that one of our suppliers may not meet our sustainability policy.",
    "The accessible entrance near the main building has been blocked for two days."
]

# ============================================================
# RUN GPT-4o TRIAGE
# ============================================================
def triage_message(message):
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0.1,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Message: {message}"}
        ]
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

print("=" * 60)
print("Q3(a) — GPT-4o ESG Triage Results")
print("=" * 60)

llm_results = []
for i, msg in enumerate(messages, 1):
    print(f"\n{'='*50}")
    print(f"Message {i}: {msg}")
    print(f"{'='*50}")
    result = triage_message(msg)
    llm_results.append({"message": msg, "classification": result})
    print(json.dumps(result, indent=2))

# Save results
with open("q3_llm_outputs.json", "w") as f:
    json.dump(llm_results, f, indent=2)

print("\n✅ GPT-4o results saved to q3_llm_outputs.json")
print("📸 Take a screenshot of this output now!")

## STEP 4 — Q3(b): Rule-Based Baseline Classifier

In [ ]:
# ============================================================
# RULE-BASED BASELINE
# ============================================================
def rule_based_classify(message):
    msg = message.lower()

    # Category detection
    if any(w in msg for w in ["water", "leak", "flood", "pipe"]):
        category = "Water"
    elif any(w in msg for w in ["recycl", "waste", "bin", "contamina"]):
        category = "Waste"
    elif any(w in msg for w in ["air condition", "energy", "electricity", "overnight", "heating"]):
        category = "Energy"
    elif any(w in msg for w in ["supplier", "procurement", "purchas"]):
        category = "Procurement"
    elif any(w in msg for w in ["access", "entrance", "blocked", "disability"]):
        category = "Accessibility"
    elif any(w in msg for w in ["policy", "governance", "compliance"]):
        category = "Governance"
    else:
        category = "Other"

    # Urgency detection
    if any(w in msg for w in ["all morning", "emergency", "immediate"]):
        urgency = "HIGH"
    elif any(w in msg for w in ["again", "overnight", "two days", "no one"]):
        urgency = "MEDIUM"
    else:
        urgency = "LOW"

    followup = "Y" if urgency in ["HIGH", "CRITICAL"] else "N"

    return {
        "issue_category": category,
        "urgency": urgency,
        "followup_required": followup,
        "method": "Rule-Based Baseline"
    }

print("=" * 60)
print("Q3(b) — Rule-Based Baseline Results")
print("=" * 60)

rule_results = []
for i, msg in enumerate(messages, 1):
    result = rule_based_classify(msg)
    rule_results.append({"message": msg, "rule_based": result})
    print(f"\nMessage {i}: {msg}")
    print(f"  Category : {result['issue_category']}")
    print(f"  Urgency  : {result['urgency']}")
    print(f"  Followup : {result['followup_required']}")

print("\n✅ Rule-based baseline complete!")
print("📸 Take a screenshot of this output now!")

## STEP 5 — Q3(b): Hugging Face Zero-Shot Classifier

In [ ]:
from transformers import pipeline

print("Loading Hugging Face model (facebook/bart-large-mnli)...")
print("First time may take 1-2 minutes to download...")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = [
    "Water issue",
    "Waste and recycling issue",
    "Energy and emissions issue",
    "Sustainability policy violation",
    "Supplier and procurement concern",
    "Accessibility and inclusion barrier",
    "Governance and compliance issue"
]

urgency_labels = [
    "Critical urgent issue requiring immediate action",
    "High priority issue requiring action within 24 hours",
    "Medium priority issue requiring action within a week",
    "Low priority informational message"
]

print("\n" + "=" * 60)
print("Q3(b) — Hugging Face Zero-Shot Results")
print("=" * 60)

hf_results = []
for i, msg in enumerate(messages, 1):
    cat_result = classifier(msg, candidate_labels)
    urg_result = classifier(msg, urgency_labels)

    hf_category = cat_result["labels"][0]
    hf_conf     = round(cat_result["scores"][0] * 100, 1)
    hf_urgency  = urg_result["labels"][0]
    hf_urg_conf = round(urg_result["scores"][0] * 100, 1)

    hf_results.append({
        "message": msg,
        "hf_category": hf_category,
        "hf_category_confidence": f"{hf_conf}%",
        "hf_urgency": hf_urgency,
        "hf_urgency_confidence": f"{hf_urg_conf}%"
    })

    print(f"\nMessage {i}: {msg}")
    print(f"  Category : {hf_category} ({hf_conf}% confidence)")
    print(f"  Urgency  : {hf_urgency} ({hf_urg_conf}% confidence)")

# Save comparison
comparison = []
for i in range(len(messages)):
    comparison.append({
        "message": messages[i],
        "gpt4o_category": llm_results[i]["classification"]["issue_category"],
        "gpt4o_urgency": llm_results[i]["classification"]["urgency"],
        "rule_based_category": rule_results[i]["rule_based"]["issue_category"],
        "rule_based_urgency": rule_results[i]["rule_based"]["urgency"],
        "hf_category": hf_results[i]["hf_category"],
        "hf_urgency": hf_results[i]["hf_urgency"]
    })

with open("q3_baseline_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("\n✅ Comparison saved to q3_baseline_comparison.json")
print("📸 Take a screenshot of this output now!")

## STEP 6 — Final Comparison Summary Table

In [ ]:
print("=" * 90)
print("FINAL COMPARISON: GPT-4o vs Rule-Based vs HuggingFace")
print("=" * 90)
print(f"{'Msg':<4} {'GPT-4o Cat':<15} {'GPT-4o Urg':<12} {'Rule Cat':<15} {'Rule Urg':<12} {'HF Cat':<20} {'HF Urg'}")
print("-" * 90)

for i, row in enumerate(comparison, 1):
    hf_cat_short = row['hf_category'].replace(' issue','').replace(' concern','').replace(' violation','').replace(' barrier','')
    hf_urg_short = row['hf_urgency'].split(' ')[0] + ' ' + row['hf_urgency'].split(' ')[1]
    print(f"{i:<4} {row['gpt4o_category']:<15} {row['gpt4o_urgency']:<12} {row['rule_based_category']:<15} {row['rule_based_urgency']:<12} {hf_cat_short:<20} {hf_urg_short}")

print("\n📸 Take a screenshot of this comparison table!")
print("📁 Upload q3_llm_outputs.json and q3_baseline_comparison.json to GitHub")